# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook guides you through loading, exploring, and analyzing the FAIR\u02c6\u00b2 tabular dataset (N=77) of cancer survivors with second primary colorectal cancer, using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset is defined by a Croissant schema and accessible via the following URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure `mlcroissant` is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the dataset object
dataset = mlc.Dataset(url)

# Access metadata as an object
meta = dataset.metadata
print(f"Dataset: {meta.name}\n\n{meta.description}\n\nVersion: {meta.version} | ID: {meta.identifier}")

## 2. Data Overview
Review available record sets, their `@id`s, fields (with `@id`s), and associated columns.

In [ ]:
# Inspect available record sets (tables) and their structure
print("Available record sets (by @id):\n")
for record_set in dataset.record_sets:
    print(f"- {record_set['@id']}  (name: {record_set.get('name','--')})")
    # List fields for this record set
    if 'field' in record_set:
        fields = record_set['field']
        if not isinstance(fields, list):
            fields = [fields]
        print("  Fields (by @id):")
        for field in fields:
            field_obj = dataset.find_by_id(field)
            if field_obj is not None:
                print(f"    - {field_obj['@id']}    (name: {field_obj.get('name', '--')})")
                # List columns for each field
                if 'column' in field_obj:
                    columns = field_obj['column']
                    if not isinstance(columns, list):
                        columns = [columns]
                    for col_id in columns:
                        col_obj = dataset.find_by_id(col_id)
                        if col_obj is not None:
                            print(f"        [column @id: {col_obj['@id']} | name: {col_obj.get('name', '--')}] ")
    print()

## 3. Data Extraction
Load data from all available record sets into DataFrames. Use exact `@id` values for referencing.

In [ ]:
# List record set @ids (from previous cell)
record_set_ids = [rs['@id'] for rs in dataset.record_sets]  # Will be filled dynamically

if not record_set_ids:
    print("No record sets found in the dataset schema. Check for correct schema structure.")
else:
    dataframes = {}
    for record_set_id in record_set_ids:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records into DataFrame for record_set '@id': {record_set_id}")
    # Example: show available columns of first record set
    example_id = record_set_ids[0]
    print(f"\nColumns in first record set ({example_id}): {dataframes[example_id].columns.tolist()}")
    display(dataframes[example_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply exploration and processing using `@id` for fields/columns. Filtering and normalization examples are performed.

In [ ]:
# For demonstration, select the first record set
record_set_id = record_set_ids[0] if record_set_ids else None
if record_set_id:
    df = dataframes[record_set_id]
    print(f"Using record set: {record_set_id}")
    # Show available numeric columns by simple inference
    numeric_cols = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    print(f"Numeric columns inferred: {numeric_cols}\n")
    if numeric_cols:
        numeric_field_id = numeric_cols[0]  # Use the first numeric column
        threshold = df[numeric_field_id].mean() if df[numeric_field_id].count() > 0 else 0

        # Filter records
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with '{numeric_field_id}' > {threshold:.2f} (first 5 rows):")
        display(filtered_df.head())

        # Normalize
        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
            filtered_df[numeric_field_id].std() if filtered_df[numeric_field_id].std() else 0)
        print(f"\nNormalized '{numeric_field_id}' (first 5 rows):")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Group by a non-numeric column if available
        group_candidates = [col for col in df.columns if col not in numeric_cols]
        if group_candidates:
            group_field = group_candidates[0]
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(f"\nGrouped data by '{group_field}' (first 5 rows):")
            display(grouped_df.head())
    else:
        print("No numeric columns available for EDA in this record set.")
else:
    print("No record set found for EDA.")

## 5. Visualization
Visualize data distributions or relationships using columns' `@id`s.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Only proceed if at least one numeric column available
if record_set_id and numeric_cols:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id], bins=15, kde=True)
    plt.title(f"Distribution of '{numeric_field_id}' in record set {record_set_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # If more than one numeric, make scatterplot
    if len(numeric_cols) > 1:
        plt.figure(figsize=(6,6))
        sns.scatterplot(x=df[numeric_cols[0]], y=df[numeric_cols[1]])
        plt.xlabel(numeric_cols[0])
        plt.ylabel(numeric_cols[1])
        plt.title(f"Scatterplot of '{numeric_cols[0]}' vs '{numeric_cols[1]}'")
        plt.show()
else:
    print("No suitable numeric fields for plotting.")

## 6. Conclusion
In this notebook, you explored the FAIR\u02c6\u00b2 colorectal cancer dataset defined by a Croissant schema using the `mlcroissant` library. You:
- Loaded the schema and metadata via URL
- Inspected available record sets and their fields and columns using their unique `@id`
- Loaded tabular records into pandas DataFrames for each record set
- Demonstrated filtering, normalization, and grouping based on selected field `@id`s
- Visualized numeric data distributions

Refer to the documentation for further details on Croissant schemas and the `mlcroissant` Python API for advanced use.